# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [38]:
# Write your code below.
%load_ext dotenv
%dotenv 


The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [39]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [40]:
import os
from glob import glob

# Write your code below.
PRICE_DATA = os.getenv("PRICE_DATA")
parquet_files = glob(os.path.join(PRICE_DATA, "**/*.parquet"), recursive = True)


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [41]:
print(parquet_files)

['../../05_src/data/prices\\A\\A_2000\\part.0.parquet', '../../05_src/data/prices\\A\\A_2001\\part.0.parquet', '../../05_src/data/prices\\A\\A_2002\\part.0.parquet', '../../05_src/data/prices\\A\\A_2003\\part.0.parquet', '../../05_src/data/prices\\A\\A_2004\\part.0.parquet', '../../05_src/data/prices\\A\\A_2005\\part.0.parquet', '../../05_src/data/prices\\A\\A_2006\\part.0.parquet', '../../05_src/data/prices\\A\\A_2007\\part.0.parquet', '../../05_src/data/prices\\A\\A_2008\\part.0.parquet', '../../05_src/data/prices\\A\\A_2009\\part.0.parquet', '../../05_src/data/prices\\A\\A_2010\\part.0.parquet', '../../05_src/data/prices\\A\\A_2011\\part.0.parquet', '../../05_src/data/prices\\A\\A_2012\\part.0.parquet', '../../05_src/data/prices\\A\\A_2013\\part.0.parquet', '../../05_src/data/prices\\A\\A_2014\\part.0.parquet', '../../05_src/data/prices\\A\\A_2015\\part.0.parquet', '../../05_src/data/prices\\A\\A_2016\\part.0.parquet', '../../05_src/data/prices\\A\\A_2017\\part.0.parquet', '../../05

In [43]:
# Write your code below.
dd_feat = dd.read_parquet(parquet_files).groupby('Ticker', group_keys=False).apply(
    lambda x: x.assign(
        Close_lag_1 = x['Close'].shift(1),
        Adj_Close_lag_1 = x['Adj Close'].shift(1),
        returns = (x['Close'] / x['Close'].shift(1)) - 1,  
        hi_lo_range = x['High'] - x['Low']
    )
)

dd_feat.compute()

C:\Users\USER\AppData\Local\Temp\ipykernel_10648\1866491237.py:2: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  dd_feat = dd.read_parquet(parquet_files).groupby('Ticker', group_keys=False).apply(


Price,Date,Adj Close,Close,High,Low,Open,Volume,Year,Close_lag_1,Adj_Close_lag_1,returns,hi_lo_range
Ticker,,,,,,,,,,,,
TXN,2012-01-03 00:00:00+00:00,20.830647,29.760000,29.940001,29.500000,29.600000,7226200.0,2012,NaN,NaN,NaN,0.440001
TXN,2012-01-04 00:00:00+00:00,20.697662,29.570000,29.750000,29.240000,29.540001,5837400.0,2012,29.760000,20.830647,-0.006384,0.510000
TXN,2012-01-05 00:00:00+00:00,20.844648,29.780001,29.900000,29.440001,29.440001,6568700.0,2012,29.570000,20.697662,0.007102,0.459999
TXN,2012-01-06 00:00:00+00:00,20.844648,29.780001,29.780001,29.459999,29.610001,6052800.0,2012,29.780001,20.844648,0.000000,0.320002
TXN,2012-01-09 00:00:00+00:00,21.110632,30.160000,30.270000,29.700001,29.719999,8040700.0,2012,29.780001,20.844648,0.012760,0.570000
...,...,...,...,...,...,...,...,...,...,...,...,...
CTAS,2022-12-23 00:00:00+00:00,110.560135,114.309998,114.372498,112.002502,112.757500,820400.0,2022,113.125000,109.414017,0.010475,2.369995
CTAS,2022-12-27 00:00:00+00:00,110.992966,114.757500,115.507500,113.985001,114.775002,1270400.0,2022,114.309998,110.560135,0.003915,1.522499
CTAS,2022-12-28 00:00:00+00:00,109.060989,112.760002,116.157501,112.739998,115.209999,1062800.0,2022,114.757500,110.992966,-0.017406,3.417503


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [44]:
# Write your code below.

dd_feat_pd = dd_feat.compute()
dd_feat_pd['ma_returns'] = dd_feat_pd['returns'].rolling(10).mean()


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return? # Yes
+ Would it have been better to do it in Dask? Why? # yes because the dataset is large and dask allows for parallel computations but the rolling operation is not available in dask so we used pandas. 

(1 pt)

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.